# Russian Lip Reading Baseline, Semester 4

Clean Colab notebook for the first PyTorch baseline on the expanded Russian lip-reading dataset.

Goal: classify a short mouth-crop video clip into one of 10 Russian word classes.

## 1. Mount Google Drive

The dataset archive is stored on Google Drive. Colab runtime storage under /content is temporary, so after a runtime reset the archive must be restored again.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Restore Dataset Archive

This cell finds colab_lipreading_dataset.zip on Google Drive and extracts it into /content/lipreading_data_fixed.

The archive was created on Windows, so we replace \ with / while extracting paths.

In [ ]:
import zipfile
from pathlib import Path
import shutil

extract_dir = Path('/content/lipreading_data_fixed')
expected_file = extract_dir / 'ru_dataset' / 'ml_splits_mouth_crops_padded_no_spk09_10' / 'speaker_top10' / 'train.csv'

if expected_file.exists():
    print('Dataset already exists:', extract_dir)
else:
    print('Dataset not found, searching zip on Google Drive...')

    matches = list(Path('/content/drive').rglob('colab_lipreading_dataset.zip'))
    print('matches:', matches)

    if len(matches) == 0:
        raise FileNotFoundError('Upload colab_lipreading_dataset.zip to Google Drive first')

    zip_path = matches[0]
    print('zip:', zip_path)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in zip_ref.infolist():
            fixed_name = member.filename.replace('\\', '/')
            target_path = extract_dir / fixed_name

            if fixed_name.endswith('/'):
                target_path.mkdir(parents=True, exist_ok=True)
                continue

            target_path.parent.mkdir(parents=True, exist_ok=True)

            with zip_ref.open(member) as source:
                with open(target_path, 'wb') as target:
                    target.write(source.read())

    print('Dataset restored:', extract_dir)

print('train.csv exists:', expected_file.exists())

## 3. Imports, Paths, Tables, Vocabulary

	rain_df, al_df, and 	est_df are pandas tables loaded from CSV files.

ocab is the list of word classes. word_to_idx converts a word into a class number.

In [ ]:
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

data_root = Path('/content/lipreading_data_fixed/ru_dataset')
split_dir = data_root / 'ml_splits_mouth_crops_padded_no_spk09_10' / 'speaker_top10'

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
test_df = pd.read_csv(split_dir / 'test.csv')

vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('train:', len(train_df))
print('val:', len(val_df))
print('test:', len(test_df))
print('vocab:', vocab)
print('device:', device)
train_df.head()

## 4. Check Class Distribution

This shows that the dataset is imbalanced: some words occur much more often than others.

In [ ]:
print('Train word counts:')
print(train_df['word'].value_counts())

print('
Validation word counts:')
print(val_df['word'].value_counts())

print('
Test word counts:')
print(test_df['word'].value_counts())

## 5. Video Loading

load_video converts one mp4 clip into a PyTorch tensor.

Output shape: [channels, frames, height, width], for example [3, 24, 96, 96].

In [ ]:
def load_video(path, num_frames=24, size=96):
    cap = cv2.VideoCapture(str(path))

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames loaded: {path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames)
    video = torch.tensor(video, dtype=torch.float32)
    video = video.permute(3, 0, 1, 2)

    return video

In [ ]:
row = train_df.iloc[0]
video_path = data_root / row['clip_path']

x = load_video(video_path)

print(video_path)
print('shape:', x.shape)
print('min/max:', x.min().item(), x.max().item())
print('word:', row['word'])
print('class id:', word_to_idx[row['word']])

## 6. PyTorch Dataset and DataLoader

Dataset returns one example: (video_tensor, class_id).

DataLoader groups examples into batches.

In [ ]:
class LipReadingDataset(Dataset):
    def __init__(self, df, data_root, word_to_idx):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.word_to_idx = word_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        video_path = self.data_root / row['clip_path']
        x = load_video(video_path)

        word = row['word']
        y = self.word_to_idx[word]
        y = torch.tensor(y, dtype=torch.long)

        return x, y

In [ ]:
train_dataset = LipReadingDataset(train_df, data_root, word_to_idx)
val_dataset = LipReadingDataset(val_df, data_root, word_to_idx)
test_dataset = LipReadingDataset(test_df, data_root, word_to_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

batch_x, batch_y = next(iter(train_loader))
print('batch_x:', batch_x.shape)
print('batch_y:', batch_y.shape)
print(batch_y)

## 7. Simple 3D-CNN Baseline

Conv3d processes video in time, height, and width.

The model outputs logits with shape [batch_size, num_classes].

In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool3d((1, 1, 1)),
            nn.Flatten(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
model = Simple3DCNN(num_classes=len(vocab)).to(device)

batch_x = batch_x.to(device)
batch_y = batch_y.to(device)

outputs = model(batch_x)
print('outputs:', outputs.shape)
print('labels:', batch_y.shape)

## 8. Train and Evaluation Functions

Training updates model weights. Evaluation only measures quality and does not update weights.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / total, correct / total

## 9. Baseline Training

This trains the simple 3D-CNN for a few epochs and saves the best checkpoint by validation accuracy.

In [ ]:
checkpoint_dir = Path('/content/drive/MyDrive/lipreading_sem4')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model = Simple3DCNN(num_classes=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'vocab': vocab,
                'word_to_idx': word_to_idx,
                'idx_to_word': idx_to_word,
                'history': history,
                'best_val_acc': best_val_acc,
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)

## 10. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.xlabel('epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='train')
plt.plot(history['val_acc'], label='val')
plt.title('Accuracy')
plt.xlabel('epoch')
plt.legend()

plt.tight_layout()
plt.show()

## 11. Prediction Distribution Diagnostics

This checks whether the model predicts many classes or collapses into one frequent class.

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(batch_y.cpu().numpy().tolist())

    return all_preds, all_labels


val_preds, val_labels = collect_predictions(model, val_loader, device)

print('pred ids:', Counter(val_preds))
print('true ids:', Counter(val_labels))

print('
pred words:')
for idx, count in Counter(val_preds).most_common():
    print(idx_to_word[idx], count)

print('
true words:')
for idx, count in Counter(val_labels).most_common():
    print(idx_to_word[idx], count)

## 12. Weighted Loss Experiment

Class weights increase the loss penalty for rare classes and reduce the penalty for frequent classes.

In [ ]:
train_counts = Counter(train_df['word'])
print(train_counts)

class_weights = []
for word in vocab:
    count = train_counts[word]
    class_weights.append(1.0 / count)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.to(device)

print(class_weights)

In [ ]:
weighted_model = Simple3DCNN(num_classes=len(vocab)).to(device)
weighted_criterion = nn.CrossEntropyLoss(weight=class_weights)
weighted_optimizer = torch.optim.Adam(weighted_model.parameters(), lr=1e-3)

num_epochs = 5
history_weighted = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_checkpoint_path = checkpoint_dir / 'simple3dcnn_weighted_best.pt'

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(weighted_model, train_loader, weighted_criterion, weighted_optimizer, device)
    val_loss, val_acc = evaluate(weighted_model, val_loader, weighted_criterion, device)

    history_weighted['train_loss'].append(train_loss)
    history_weighted['train_acc'].append(train_acc)
    history_weighted['val_loss'].append(val_loss)
    history_weighted['val_acc'].append(val_acc)

    print(
        f'epoch {epoch + 1}/{num_epochs} | '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
        f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}'
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch + 1,
                'model_state_dict': weighted_model.state_dict(),
                'optimizer_state_dict': weighted_optimizer.state_dict(),
                'vocab': vocab,
                'history': history_weighted,
                'best_val_acc': best_val_acc,
                'class_weights': class_weights.detach().cpu(),
            },
            best_checkpoint_path,
        )
        print('saved best:', best_checkpoint_path)

## 13. Next Experiments

Recommended next steps:

1. Use WeightedRandomSampler or oversampling for rare classes.
2. Add per-class accuracy and confusion matrix.
3. Compare full-frame clips and mouth crops.
4. Add more clean speakers and balance word counts.
5. Try a stronger temporal model: CNN+GRU/LSTM or 3D-CNN+Transformer.